# Lesson 4.5 — Designing Efficient Batch Processing Strategies

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- State Message Batches API tradeoffs: 50% savings, up to 24h window, no latency SLA
- Match batch vs synchronous API to workload latency requirements
- Correlate requests with custom_id and resubmit only failed items
- Refine prompts on a sample before batch-processing large volumes

**You will produce:** A batch submission builder with custom_id plus a failure-resubmission strategy.

## Concepts

The **Message Batches API** gives **50% cost savings** with processing **up to 24 hours** and **no guaranteed latency SLA**. It fits **non-blocking, latency-tolerant** workloads (overnight reports, weekly audits, nightly test generation) and is **wrong for blocking** workflows (pre-merge checks). Sample Questions 10-11 reinforce this.

Key constraints & patterns:
- The batch API does **not** support multi-turn tool calling within a single request (can't execute tools mid-request and return results).
- **`custom_id`** correlates request/response pairs.
- **Handle failures** by resubmitting only the failed `custom_id`s, with modifications (e.g. chunking documents that exceeded context limits).
- **Compute submission frequency** from SLA math (e.g. 4-hour windows to guarantee a 30-hour SLA given 24-hour processing).
- **Refine prompts on a sample** before batch-processing large volumes to maximize first-pass success and reduce resubmission costs.

> **Reference:** [Anthropic - Message Batches API](https://docs.anthropic.com/en/docs/build-with-claude/batch-processing)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Match API to workload

Blocking pre-merge -> synchronous; overnight audit -> batch.

In [ ]:
def choose_api(workload):
    return "batch" if workload.get("latency_tolerant") and not workload.get("blocking") \
        else "synchronous"

print("pre-merge check:", choose_api({"blocking": True}))
print("overnight tech-debt report:", choose_api({"latency_tolerant": True}))

## Code-Along: Step 2 — Build batch requests with custom_id

Each request carries a custom_id so we can correlate and resubmit precisely.

In [ ]:
def build_batch(documents):
    return [{
        "custom_id": f"doc-{i}",
        "params": {
            "model": MODEL_HAIKU,
            "max_tokens": 512,
            "messages": [{"role": "user", "content": d}],
        },
    } for i, d in enumerate(documents)]

batch = build_batch(["doc A text", "doc B text", "doc C text"])
print(batch[0]['custom_id'], '->', list(batch[0]['params'].keys()))
# Submit with client.messages.batches.create(requests=batch)  (needs API key)

## Code-Along: Step 3 — Resubmit only failed items (with chunking)

Failures identified by custom_id get modified and resubmitted - not the whole batch.

In [ ]:
def resubmit_failures(results, documents):
    retry = []
    for r in results:
        if r["status"] == "errored":
            idx = int(r["custom_id"].split("-")[1])
            doc = documents[idx]
            if r.get("error") == "context_length_exceeded":
                half = len(doc) // 2
                retry.append(doc[:half]); retry.append(doc[half:])  # chunk
            else:
                retry.append(doc)
    return build_batch(retry)

fake_results = [{"custom_id": "doc-1", "status": "errored",
                 "error": "context_length_exceeded"},
                {"custom_id": "doc-0", "status": "succeeded"}]
print('resubmission size:', len(resubmit_failures(fake_results, ["A", "B"*10, "C"])) )

## Exercise

Write `submission_interval(processing_hours, sla_hours)` returning the max gap between batch submissions to meet the SLA (e.g. 24h processing, 30h SLA -> <=6h; the guide's 4h window is a safe choice). Then assert a blocking pre-merge workload is never routed to batch, and note why multi-turn tool calling can't run inside a batch request.

In [ ]:
# TODO: submission_interval(processing_hours, sla_hours) -> hours
# TODO: assert choose_api({'blocking': True}) == 'synchronous'

## Key Takeaways

- Message Batches API: 50% cheaper, up to 24h, no latency SLA - for non-blocking work only.
- Use custom_id to correlate pairs and resubmit only failed items (chunk oversized docs).
- Batch requests can't do multi-turn tool calling mid-request.
- Refine prompts on a sample before processing large volumes to cut resubmission costs.

---

**Next:** [Lesson 4.6 — Multi-Instance & Multi-Pass Review](./lesson-4.6-multi-pass-review.ipynb)